## Boosting

#### CatBoost

In [17]:
# --- 1. Importer nødvendige pakker ---
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import matplotlib.pyplot as plt
import numpy as np

# --- 2. Les inn dimensjonsredusert data ---
df = pd.read_csv('cyberbullying_datasets/course_dataset/cb_pca_compressed.csv.gz')
print("✅ Data loaded:", df.shape)

X = df.drop("bullying", axis=1)
y = df["bullying"]

# --- 3. Split i trenings- og testsett ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- 4. Opprett CatBoost-modellen ---
cat_model = CatBoostClassifier(
    iterations=500,          # hvor mange trær som bygges (øker nøyaktighet, men også tid)
    depth=6,                 # tre-dybde (balanse mellom kompleksitet og generalisering)
    learning_rate=0.05,      # hvor mye hvert tre "lærer" per steg
    loss_function='Logloss', # binær klassifisering
    eval_metric='Accuracy',
    random_seed=42,
    verbose=100              # skriver status for hver 100 iterasjon
)

# --- 5. Tren modellen ---
cat_model.fit(X_train, y_train, eval_set=(X_test, y_test))

# --- 6. Evaluer på testsett ---
y_pred = cat_model.predict(X_test)

✅ Data loaded: (11069, 5054)
0:	learn: 0.6293619	test: 0.6323397	best: 0.6323397 (0)	total: 397ms	remaining: 3m 18s
100:	learn: 0.7506494	test: 0.6869919	best: 0.6878952 (94)	total: 33.2s	remaining: 2m 11s
200:	learn: 0.8247318	test: 0.6928636	best: 0.6969286 (197)	total: 1m 7s	remaining: 1m 40s
300:	learn: 0.8997177	test: 0.7059621	best: 0.7064137 (299)	total: 1m 45s	remaining: 1m 9s
400:	learn: 0.9496330	test: 0.7073171	best: 0.7104788 (340)	total: 2m 23s	remaining: 35.5s
499:	learn: 0.9721062	test: 0.7086721	best: 0.7127371 (409)	total: 3m 6s	remaining: 0us

bestTest = 0.7127371274
bestIteration = 409

Shrink model to first 410 iterations.


In [19]:
print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Accuracy: 0.7127371273712737

Classification Report:
               precision    recall  f1-score   support

           0       0.69      0.76      0.72      1105
           1       0.73      0.67      0.70      1109

    accuracy                           0.71      2214
   macro avg       0.71      0.71      0.71      2214
weighted avg       0.71      0.71      0.71      2214



#### XGboost

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, accuracy_score
from xgboost import XGBClassifier


df_pca = pd.read_csv("cyberbullying_datasets/course_dataset/cb_pca_compressed.csv.gz")


X = df_pca.drop(columns=["bullying"])
y = df_pca["bullying"]

# train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)


xgb_model = XGBClassifier(
    n_estimators=500,       # number of boosting rounds
    learning_rate=0.01,     # smaller = slower but better generalization
    max_depth=5,            # depth of each tree
    subsample=0.8,          # randomly sample training data (helps avoid overfitting)
    colsample_bytree=0.8,   # randomly sample features
    random_state=0,
    use_label_encoder=False,
    eval_metric='logloss'
)

# # K-fold cross validation 
# cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# scores = []

# for train_idx, test_idx in cv.split(X, y):
#     X_train_cv, X_test_cv = X.iloc[train_idx], X.iloc[test_idx]
#     y_train_cv, y_test_cv = y.iloc[train_idx], y.iloc[test_idx]

#     xgb_model.fit(X_train_cv, y_train_cv)
#     y_pred_cv = xgb_model.predict(X_test_cv)
#     scores.append(accuracy_score(y_test_cv, y_pred_cv))

# print(f"Cross-validation accuracies: {scores}")
# print(f"Mean CV accuracy: {np.mean(scores):.4f}")

xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_test)

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


C:\Users\David\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\xgboost\core.py:158: UserWarning: [12:18:34] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)



Accuracy: 0.6937669376693767

Classification Report:
               precision    recall  f1-score   support

           0       0.67      0.75      0.71      1105
           1       0.72      0.64      0.68      1109

    accuracy                           0.69      2214
   macro avg       0.70      0.69      0.69      2214
weighted avg       0.70      0.69      0.69      2214



#### SE BORT IFRA UNDER

In [1]:
from sklearn.model_selection import GridSearchCV

params = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [100, 300, 500],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0]
}

grid = GridSearchCV(xgb_model, params, cv=3, scoring='accuracy', verbose=2, n_jobs=-1)
grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best CV accuracy:", grid.best_score_)


NameError: name 'xgb_model' is not defined

In [10]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from xgboost import XGBClassifier
import numpy as np
import pandas as pd

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.8, 1.0],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

for n in param_grid["n_estimators"]:
    for d in param_grid["max_depth"]:
        for lr in param_grid["learning_rate"]:
            for sub in param_grid["subsample"]:
                xgb_model = XGBClassifier(
                    n_estimators=n,
                    max_depth=d,
                    learning_rate=lr,
                    subsample=sub,
                    colsample_bytree=0.8,
                    random_state=42,
                    use_label_encoder=False,
                    eval_metric="logloss"
                )


                xgb_model.fit(X_train, y_train)
                y_pred = xgb_model.predict(X_test)

                results.append({
                    "n_estimators": n,
                    "max_depth": d,
                    "learning_rate": lr,
                    "subsample": sub,
                })
                print(f"n={n}, depth={d}, lr={lr}, subsample={sub}")

results_df = pd.DataFrame(results)
best_params = results_df.loc[results_df['mean_f1'].idxmax()]
print("\nBest parameters:")
print(best_params)


C:\Users\David\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\xgboost\core.py:158: UserWarning: [15:19:38] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


n=100, depth=3, lr=0.01, subsample=0.8


C:\Users\David\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\xgboost\core.py:158: UserWarning: [15:20:11] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


n=100, depth=3, lr=0.01, subsample=1.0


C:\Users\David\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\xgboost\core.py:158: UserWarning: [15:20:44] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


...

C:\Users\David\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\xgboost\core.py:158: UserWarning: [23:05:52] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


n=300, depth=7, lr=0.1, subsample=0.8


C:\Users\David\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\xgboost\core.py:158: UserWarning: [01:15:38] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


n=300, depth=7, lr=0.1, subsample=1.0


KeyError: 'mean_f1'

## Bagging

#### SVM BAGGING

In [6]:
from sklearn.ensemble import BaggingClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score


df_pca = pd.read_csv("cyberbullying_datasets/course_dataset/cb_pca_compressed.csv.gz")


X = df_pca.drop(columns=["bullying"])
y = df_pca["bullying"]

# train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

bagged_svm = BaggingClassifier(
    estimator=SVC(kernel="rbf", C=1.0,gamma="scale"),
    n_estimators=20,
    random_state=0
).fit(X_train, y_train)

y_pred = bagged_svm.predict(X_test)

In [7]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.7186088527551943

Classification Report:
               precision    recall  f1-score   support

           0       0.70      0.77      0.73      1105
           1       0.75      0.66      0.70      1109

    accuracy                           0.72      2214
   macro avg       0.72      0.72      0.72      2214
weighted avg       0.72      0.72      0.72      2214

